In [1]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [2]:
df = pd.read_csv("final_resume_dataset.csv")

In [3]:
print("Unique Titles:", df["title"].nunique())

print(df["title"].value_counts().head(20))

Unique Titles: 16187
title
Network Administrator                                                                        624
Oracle Database Administrator Oracle Database Administrator                                  282
Python Developer Python Developer Python Developer Python Developer                          234
Python Developer Python Developer Python Developer                                           195
Database Administrator                                                                       186
Front End Developer                                                                          186
Java Developer Java Developer Java Developer                                                 117
Database Administrator Database Administrator                                                108
Python Developer Python Developer Python Developer Python Developer Python Developer         105
Java Developer                                                                               102
Ora

In [4]:
job_roles = [

    "Python Developer",
    "Java Developer",
    "Full Stack Java Developer",
    "Full Stack Developer",

    "Front End Developer",
    "Front End Web Developer",
    "Front-end Developer",

    "Database Administrator",
    "Oracle Database Administrator",
    "SQL Database Administrator",
    "SQL Server Database Administrator",

    "Network Administrator",
    "Network Engineer",

    "IT Project Manager",
    "Project Manager",

    "IT Security Analyst",

    "Software Engineer",
    "Data Scientist",
    "Data Analyst",
    "Machine Learning Engineer",

    "DevOps Engineer",
    "Cloud Engineer",

    "Business Analyst",
    "Web Developer",

    "Backend Developer",
    "Frontend Developer",

    "QA Engineer",
    "Test Engineer",

    "System Administrator"
]

In [5]:
def normalize_text(text):

    text = str(text)

    text = text.lower()

    text = text.replace("-", " ")

    text = text.replace("/", " ")

    text = text.replace("|", " ")

    text = re.sub(r"[^a-z ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
normalized_roles = {}

for role in job_roles:

    normalized_roles[normalize_text(role)] = role

In [7]:
def extract_role(title):

    title = normalize_text(title)

    matches = []

    for role in normalized_roles:

        if role in title:
            matches.append(role)

    if len(matches) == 0:
        return "Other"

    best = max(matches, key=len)

    return normalized_roles[best]

In [8]:
df["title"] = df["title"].apply(extract_role)

In [9]:

print(df["title"].value_counts())

title
Other                                8367
Network Administrator                7104
Front-end Developer                  5688
Database Administrator               5481
IT Project Manager                   4077
Java Developer                       3960
Project Manager                      3771
IT Security Analyst                  2382
Python Developer                     2244
Front End Web Developer              1956
Oracle Database Administrator        1890
Full Stack Developer                 1332
System Administrator                 1260
Software Engineer                    1197
Business Analyst                      975
Network Engineer                      732
Full Stack Java Developer             708
SQL Server Database Administrator     483
Web Developer                         462
SQL Database Administrator            447
Data Analyst                          165
Backend Developer                      72
DevOps Engineer                        54
Test Engineer               

In [10]:
print(df["title"].value_counts()["Other"])

8367


In [11]:

df = df[df["title"] != "Other"]

In [12]:
print(df["title"].nunique())

print(df["title"].value_counts())

28
title
Network Administrator                7104
Front-end Developer                  5688
Database Administrator               5481
IT Project Manager                   4077
Java Developer                       3960
Project Manager                      3771
IT Security Analyst                  2382
Python Developer                     2244
Front End Web Developer              1956
Oracle Database Administrator        1890
Full Stack Developer                 1332
System Administrator                 1260
Software Engineer                    1197
Business Analyst                      975
Network Engineer                      732
Full Stack Java Developer             708
SQL Server Database Administrator     483
Web Developer                         462
SQL Database Administrator            447
Data Analyst                          165
Backend Developer                      72
DevOps Engineer                        54
Test Engineer                          48
Frontend Developer       

In [13]:
df.to_csv("../dataset/clean_resume_dataset.csv", index=False)

print("Dataset Saved Successfully")

Dataset Saved Successfully


In [14]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [15]:
df = pd.read_csv("../dataset/clean_resume_dataset.csv")

print(df.shape)

df.head()

(46566, 11)


,person_id,name,email,phone,linkedin,skill,program,title,firm,resume_text,ability
0,1,database administrator,NaN,NaN,NaN,Database administration Database Ms sql server...,Bachelor of Science,Database Administrator,Family Private Care LLC Incomm,database administration database ms sql server...,Installation and Building Server Running Backu...
1,2,database administrator,NaN,NaN,NaN,sql server management studio visual studio sql...,bsc in computer science,Database Administrator,Intercontinental Registry,sql server management studio visual studio sql...,database management systems administration dev...
2,3,oracle database administrator,NaN,NaN,NaN,DATABASES ORACLE (4 years) ORACLE 10G SQL LINU...,Master of Computer Applications in Science and...,Oracle Database Administrator,Cognizant Convergys,databases oracle 4 years oracle 10g sql linux ...,Over 4+ years of Experience as Architecture Ex...
3,4,amazon redshift administrator and etl develope...,NaN,NaN,NaN,Maintain multiple database environments (Redsh...,Bachelor in Computer Science,Database Administrator,"MSP Recovery - Fort Lauderdale, FL CEAACES - Q...",maintain multiple database environments redshi...,SQL management PostgresSQL Oracle MySQL micros...
4,5,scrum master scrum master scrum master,NaN,NaN,NaN,Scrum Agile software development Product backl...,NaN,Oracle Database Administrator,Quest Technologies Prudential Time Warner Cable,scrum agile software development product backl...,Scrum Master Agile software development Produc...


In [16]:
X = df["resume_text"]

y = df["title"]

In [17]:

title_mapping = {
    "Front-end Developer": "Front End Developer",
    "Frontend Developer": "Front End Developer",
    "Front End Web Developer": "Front End Developer",

    "Web Developer": "Front End Developer",

    "SQL Server Database Administrator": "Database Administrator",
    "SQL Database Administrator": "Database Administrator",
    "Oracle Database Administrator": "Database Administrator",

    "Network Engineer": "Network Administrator",

    "Full Stack Java Developer": "Full Stack Developer",

    "System Administrator": "Network Administrator"
}

df["title"] = df["title"].replace(title_mapping)

In [18]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("Number of Classes:", len(encoder.classes_))

Number of Classes: 28


In [19]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
tfidf = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    strip_accents="unicode",
    sublinear_tf=True,
    max_features=8000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.80
)

X_train = tfidf.fit_transform(X_train_text)

X_test = tfidf.transform(X_test_text)

In [21]:
selector = SelectKBest(
    score_func=chi2,
    k=min(3000, X_train.shape[1])
)

X_train = selector.fit_transform(X_train, y_train)

X_test = selector.transform(X_test)

In [22]:
param_grid = {

    "C":[0.01,0.05,0.1,0.5,1],

    "solver":["lbfgs"],

    "penalty":["l2"],

    "class_weight":[None,"balanced"],

    "max_iter":[5000]
}

In [23]:
grid = GridSearchCV(

    LogisticRegression(random_state=42),

    param_grid = {
    "C": [0.1, 1, 10],
    "solver": ["lbfgs"],
    "penalty": ["l2"]
}
)


grid.fit(X_train, y_train)

model = grid.best_estimator_

e:\final_infosys\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
e:\final_infosys\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
e:\final_infosys\.venv\Lib\site-packages\sklearn\linear_model\_l

In [24]:
print("Best Parameters")

print(grid.best_params_)

print()

print("Cross Validation Accuracy")

print(grid.best_score_)

Best Parameters
{'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}

Cross Validation Accuracy
0.914286663922414


In [25]:
train_pred = model.predict(X_train)

test_pred = model.predict(X_test)

In [26]:
train_accuracy = accuracy_score(
    y_train,
    train_pred
)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

print("Training Accuracy :", train_accuracy)

print("Testing Accuracy :", test_accuracy)

Training Accuracy : 0.9584183399549018
Testing Accuracy : 0.9234485720420872


In [27]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/career_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [28]:
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

['models/tfidf_vectorizer.pkl']

In [29]:
import joblib

joblib.dump(model, "../models/career_model.pkl")
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(selector, "../models/feature_selector.pkl")
joblib.dump(encoder, "../models/label_encoder.pkl")

print("All files saved successfully!")

All files saved successfully!


In [30]:
import pdfplumber
import re
from docx import Document

In [31]:
def extract_pdf_text(path):

    text = ""

    with pdfplumber.open(path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:

                text += page_text + "\n"

    return text

In [32]:
def extract_docx_text(path):

    doc = Document(path)

    text = "\n".join(
        para.text for para in doc.paragraphs
    )

    return text

In [33]:
def clean_resume(text):

    text = text.lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"\S+@\S+", " ", text)

    text = re.sub(r"\d+", " ", text)

    text = re.sub(r"[^a-z ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [34]:
resume_path = r"E:\final_infosys\final_infosys\uploads\_Sudarshan_Resume_i-exceed[1].pdf"

resume = extract_pdf_text(resume_path)

resume = clean_resume(resume)

In [35]:
import os

resume_path = r"E:\final_infosys\final_infosys\uploads\_Sudarshan_Resume_i-exceed[1].pdf"

extension = os.path.splitext(resume_path)[1].lower()

if extension == ".pdf":
    resume = extract_pdf_text(resume_path)
elif extension == ".docx":
    resume = extract_docx_text(resume_path)
else:
    raise ValueError("Unsupported file format")

resume = clean_resume(resume)

In [36]:
resume_vector = tfidf.transform([resume])

In [37]:

resume_vector = selector.transform(resume_vector)

In [38]:
import numpy as np

probabilities = model.predict_proba(resume_vector)[0]

top3 = np.argsort(probabilities)[::-1][:3]

In [39]:
print("=" * 50)
print("Top 3 Career Predictions")
print("=" * 50)

for rank, idx in enumerate(top3, start=1):

    role = encoder.inverse_transform([idx])[0]

    confidence = probabilities[idx] * 100

    print(f"{rank}. {role:30} {confidence:.2f}%")

Top 3 Career Predictions
1. Full Stack Developer           77.85%
2. Java Developer                 4.73%
3. Project Manager                3.71%


In [40]:
prediction = model.predict(resume_vector)

print(
    encoder.inverse_transform(prediction)[0]
)

Full Stack Developer


In [41]:
# ============================================================
# FINAL LOGISTIC REGRESSION METRICS
# ============================================================

import os
import json
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# ------------------------------------------------------------
# IMPORTANT:
# Use the variable containing your TRAINED Logistic Regression
# model here.
# ------------------------------------------------------------

lr_model = model

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

lr_train_acc = accuracy_score(
    y_train,
    y_train_pred
)

lr_test_acc = accuracy_score(
    y_test,
    y_test_pred
)

lr_balanced_acc = balanced_accuracy_score(
    y_test,
    y_test_pred
)

lr_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

lr_weighted_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted"
)

lr_overfit_gap = lr_train_acc - lr_test_acc

# ------------------------------------------------------------
# Print
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FINAL LOGISTIC REGRESSION")
print("=" * 55)

print(
    f"Training Accuracy   : {lr_train_acc * 100:.2f}%"
)

print(
    f"Testing Accuracy    : {lr_test_acc * 100:.2f}%"
)

print(
    f"Balanced Accuracy   : {lr_balanced_acc * 100:.2f}%"
)

print(
    f"Macro F1-Score      : {lr_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1-Score   : {lr_weighted_f1 * 100:.2f}%"
)

print(
    f"Overfit Gap         : {lr_overfit_gap * 100:.2f}%"
)


FINAL LOGISTIC REGRESSION
Training Accuracy   : 95.84%
Testing Accuracy    : 92.34%
Balanced Accuracy   : 83.13%
Macro F1-Score      : 86.15%
Weighted F1-Score   : 92.22%
Overfit Gap         : 3.50%


In [42]:
# ============================================================
# SAVE LOGISTIC REGRESSION METRICS
# ============================================================

MODEL_DIR = r"E:\final_infosys\final_infosys\models"

os.makedirs(MODEL_DIR, exist_ok=True)

METRICS_PATH = os.path.join(
    MODEL_DIR,
    "model_metrics.json"
)

# Load existing RF + XGBoost metrics if available
if os.path.exists(METRICS_PATH):

    with open(METRICS_PATH, "r") as f:
        metrics = json.load(f)

else:
    metrics = {}

# Add / replace Logistic Regression
metrics["Logistic Regression"] = {
    "training_accuracy": float(lr_train_acc),
    "testing_accuracy": float(lr_test_acc),
    "balanced_accuracy": float(lr_balanced_acc),
    "macro_f1": float(lr_macro_f1),
    "weighted_f1": float(lr_weighted_f1)
}

# Save
with open(METRICS_PATH, "w") as f:
    json.dump(
        metrics,
        f,
        indent=4
    )

print("\n✅ model_metrics.json updated")
print("\nFinal metrics:")
print(json.dumps(metrics, indent=4))


✅ model_metrics.json updated

Final metrics:
{
    "Random Forest": {
        "training_accuracy": 0.9663,
        "testing_accuracy": 0.9247,
        "balanced_accuracy": 0.864
    },
    "XGBoost": {
        "training_accuracy": 0.9487,
        "testing_accuracy": 0.9307,
        "balanced_accuracy": 0.8533
    },
    "Logistic Regression": {
        "training_accuracy": 0.9584183399549018,
        "testing_accuracy": 0.9234485720420872,
        "balanced_accuracy": 0.8313167782354657,
        "macro_f1": 0.8614849333523169,
        "weighted_f1": 0.9221529881043937
    }
}
